# Dataset 3: Wednesday-workingHours EDA (DoS Family & Heartbleed)
This notebook analyzes Denial-of-Service attacks (Hulk, GoldenEye, slowloris, Slowhttptest) and the Heartbleed exploit.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Wednesday-workingHours.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

print(f"Loaded Wednesday: {df.shape[0]:,} rows")
print(f"Class breakdown:\n{df['Label'].value_counts()}")


## 1. DoS Flooding vs Slow-and-Low Mechanics & Heartbleed Profile

In [ ]:
for c in [c for c in df.columns if c != 'Label']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

hb_df = df[df['Label'] == 'Heartbleed']
print(f"Heartbleed (11 flows) Mean Backward Bytes: {hb_df['Total Length of Bwd Packets'].mean():,.2f} bytes")
print(f"Heartbleed Destination Port: {hb_df['Destination Port'].value_counts().to_dict()}")

plt.figure(figsize=(10, 4.5))
sns.barplot(x=df['Label'].value_counts().index, y=df['Label'].value_counts().values, palette='tab10')
plt.yscale('log')
plt.xticks(rotation=20, ha='right')
plt.title("Wednesday: Class Distribution (Log Scale)")
plt.show()


## 2. Key Findings & Preprocessing Recommendations
- 100% of DoS attacks target Port 80; Heartbleed targets Port 444.
- DoS Slowhttptest has median 0 backward packets (unidirectional request starvation).
- Heartbleed is an extreme minority class (11 flows) requiring stratified evaluation.
